# ViArc VSM AGI-2 — SFT iteration 1 (LoRA r2)

**Purpose:** fine-tune the grid proposer (base: Qwen3-4B-Instruct,
Apache-2.0) on ARC-AGI-2 training pairs with D4 augmentation + per-task
leave-one-out prompts (answer-leak free), LoRA r=16 α=32 on attn q/k/v/o,
seq 512 (v12: T4-validated — 768 OOMed at loss upcast), lr 1e-4 —
spec from `config/harness_config.json` (single source of truth).
v12 CALIB (S5 2026-10-19): max_steps=150 — adapter candidato +
curva de loss + velocidad; corpus completo post-entry.

**Data:** competition `arc-agi_training_challenges(+solutions).json`
(train split only, by-task hash 80/20 — heldout never trained).

**Output:** `adapter_r2/` (adapter + `TRAIN_META.json` provenance) →
staged as a private dataset; gate = held-out p1 ≥ zero-shot + 2pp
(evaluated by `bench_local.py` with `LLM_ADAPTER_PATH`, separate run).

**Reproduce:** run all cells (GPU, internet OFF — private training kernel,
never submitted to the competition). Deterministic seed 42 (R10).


In [ ]:
import json, os, sys, glob

os.environ.setdefault("PYTHONHASHSEED", "0")
# v7 (v6 incident: CUDA OOM on T4 at first backward): tame fragmentation
# before torch initializes the allocator.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# v10 incident (v9 logs): worker exposed 2 GPUs → transformers wrapped
# the model in nn.DataParallel (replicate/gather traffic blew the
# 14.56GiB budget). All our memory math assumed ONE GPU: pin it here,
# before torch first touches CUDA.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ── 1. peft (preinstalled — kernel runs internet OFF) ────────────
try:
    import peft  # noqa: F401
    print("peft ok:", peft.__version__)
except ImportError as e:
    raise SystemExit(
        f"peft missing offline ({e}) — image should ship it (v1 did)")

# v4 incident: image ships torchao 0.10.0 but peft 0.20 REQUIRES
# >0.16 — its LoRA dispatcher RAISES on old versions. LoRA never
# needs torchao: remove it (offline-safe: pip uninstall needs no
# network) and belt-and-suspenders patch the availability probes.
try:
    import subprocess
    subprocess.check_call(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("torchao uninstalled")
except Exception as e:
    print("torchao uninstall skipped:", e)
sys.modules.pop("torchao", None)
try:
    import peft.import_utils as _piu
    _piu.is_torchao_available = lambda: False
    from peft.tuners.lora import torchao as _ptao
    _ptao.is_torchao_available = lambda: False
    print("torchao probes patched")
except Exception as e:
    print("torchao probe patch skipped:", e)

# ── 2. modules + config (embedded) ───────────────────────────────
MODULES = {"sft_prepare.py": "#!/usr/bin/env python3\n# design: projects/arc-prize/knowledge/viarc_master_plan.vsm\n\"\"\"sft_prepare.py \u2014 deterministic SFT dataset builder (D4 augment).\n\n    python3 sft_prepare.py [--data DIR] [--out DIR] [--n-variants 7] [--seed 42]\n\nBuilds chat-format jsonl for LoRA fine-tuning of the grid proposer:\n  {\"messages\": [\n      {\"role\": \"user\",      \"content\": <grid prompt>},\n      {\"role\": \"assistant\", \"content\": <output grid text>} ]}\n\nAugment: D4 group (identity, flip_h, flip_v, rot90, rot180, rot270,\ntranspose) + deterministic recolor per variant. Split is BY TASK hash\n(80/20, no pair-level leakage). Stdlib-only (pure python transforms),\ndeterministic (R10): same seed+data -> byte-identical output.\n\nRuns on LXC 114 (CPU lab) and inside the Kaggle kernel (in-kernel prep\nis possible but unnecessary \u2014 generated artifact is small text jsonl).\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport glob\nimport hashlib\nimport json\nimport os\nimport random\nimport sys\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\nfrom llm_serialize import build_prompt, grid_to_text  # noqa: E402\n\n\n# \u2500\u2500 D4 transforms (pure python, list-of-lists) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef flip_h(g):\n    return [row[::-1] for row in g]\n\n\ndef flip_v(g):\n    return g[::-1]\n\n\ndef rot90(g):\n    return [list(r) for r in zip(*g[::-1])]\n\n\ndef rot180(g):\n    return [row[::-1] for row in g[::-1]]\n\n\ndef rot270(g):\n    return [list(r) for r in zip(*g)][::-1]\n\n\ndef transpose(g):\n    return [list(r) for r in zip(*g)]\n\n\nVARIANTS = [\n    (\"id\", lambda g: [row[:] for row in g]),\n    (\"fh\", flip_h), (\"fv\", flip_v),\n    (\"r90\", rot90), (\"r180\", rot180), (\"r270\", rot270),\n    (\"t\", transpose),\n]\n\n\ndef recolor(g, perm):\n    return [[perm[v] for v in row] for row in g]\n\n\ndef make_perm(seed: int):\n    rng = random.Random(seed)\n    digits = list(range(10))\n    rng.shuffle(digits)\n    return digits\n\n\n# \u2500\u2500 Task iteration + split \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef task_split(task_id: str, seed: int) -> str:\n    h = int(hashlib.sha256(f\"{seed}:{task_id}\".encode()).hexdigest(), 16)\n    return \"train\" if (h % 10) < 8 else \"heldout\"\n\n\ndef build(data_dir: str, out_dir: str, n_variants: int, seed: int) -> dict:\n    paths = sorted(glob.glob(os.path.join(data_dir, \"*.json\")))\n    if not paths:\n        raise SystemExit(f\"no puzzles under {data_dir}\")\n    os.makedirs(out_dir, exist_ok=True)\n    writers = {\n        \"train\": open(os.path.join(out_dir, \"train.jsonl\"), \"w\"),\n        \"heldout\": open(os.path.join(out_dir, \"heldout.jsonl\"), \"w\"),\n    }\n    stats = {\"examples\": {}, \"tasks\": {}}\n    try:\n        for pi, path in enumerate(paths):\n            tid = os.path.splitext(os.path.basename(path))[0]\n            split = task_split(tid, seed)\n            stats[\"tasks\"][split] = stats[\"tasks\"].get(split, 0) + 1\n            with open(path) as f:\n                task = json.load(f)\n            pairs = [(p[\"input\"], p[\"output\"])\n                     for p in task.get(\"train\", [])]\n            pairs += [(p[\"input\"], p[\"output\"])\n                      for p in task.get(\"test\", []) if \"output\" in p]\n            if not pairs:\n                continue\n            for vi, (vname, fn) in enumerate(VARIANTS[:n_variants]):\n                # deterministic per-task, per-variant recolor\n                perm = make_perm(seed ^ (pi * 131) ^ (vi * 17))\n                aug = []\n                for inp, out in pairs:\n                    ti, to = fn(inp), fn(out)\n                    if vname != \"id\":\n                        ti, to = recolor(ti, perm), recolor(to, perm)\n                    aug.append((ti, to))\n                # leave-one-out per pair: the shown train examples never\n                # include the pair being predicted (no answer leak)\n                for idx, (ti, to) in enumerate(aug):\n                    shown = aug[:idx] + aug[idx + 1:]\n                    rec = {\n                        \"messages\": [\n                            {\"role\": \"user\",\n                             \"content\": build_prompt(shown, ti)},\n                            {\"role\": \"assistant\",\n                             \"content\": grid_to_text(to)},\n                        ],\n                        \"task\": tid, \"variant\": vname,\n                    }\n                    writers[split].write(json.dumps(rec) + \"\\n\")\n                    stats[\"examples\"][split] = \\\n                        stats[\"examples\"].get(split, 0) + 1\n    finally:\n        for w in writers.values():\n            w.close()\n    return stats\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--data\", default=os.path.join(HERE, \"..\", \"data\",\n                                                   \"training\"))\n    ap.add_argument(\"--out\", default=os.path.join(HERE, \"..\", \"data\", \"sft\"))\n    ap.add_argument(\"--n-variants\", type=int, default=7)\n    ap.add_argument(\"--seed\", type=int, default=42)\n    args = ap.parse_args()\n    stats = build(args.data, args.out, args.n_variants, args.seed)\n    print(json.dumps(stats, indent=2))\n    print(\"wrote:\", os.path.join(args.out, \"train.jsonl\"),\n          \"+ heldout.jsonl\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "sft_train.py": "#!/usr/bin/env python3\n# design: projects/arc-prize/knowledge/viarc_master_plan.vsm\n\"\"\"sft_train.py \u2014 LoRA fine-tuner for the ViArc-2 grid proposer (r2).\n\n    SFT_BASE=/kaggle/input/qwen3-4b-instruct \\\n    SFT_DATA=/kaggle/working/data/sft \\\n    python3 sft_train.py [--config config/harness_config.json] [--out OUT]\n\nConfig-driven from harness_config.json \u2192 sft section (single source of\ntruth; edit there, not here):\n  LoRA r/alpha/dropout, target attn q/k/v/o, seq_len, lr, epochs,\n  per-device bs \u00d7 grad-accum, warmup, precision auto (bf16 on Ampere+,\n  fp16 on T4/Turing), seed.\n\nOutputs (--out, default $SFT_OUT or ./adapter_r2):\n  adapter_config.json + adapter_model.safetensors + TRAIN_META.json\n  (provenance: base, config hash, data stats, final losses) \u2014 staged as\n  the r2_sft_adapter dataset for kernel mount.\n\nR10: fixed seed, deterministic dataloading order (sorted jsonl), no RNG\nin eval; bitwise reproducibility across GPU archs is NOT claimed (tactic\nlayer, R10.1) \u2014 the gate is the held-out p1 benchmark, not bit equality.\nRequires: torch, transformers, peft, datasets not required (stdlib jsonl).\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport sys\nimport time\n\n# v12 (smoke incident Oct19): allocator fragmentation insurance \u2014 must be\n# set BEFORE torch first touches CUDA. The kernel RUN cell already sets it\n# (v7); defense-in-depth for local/other-driver runs of this script.\nos.environ.setdefault(\"PYTORCH_CUDA_ALLOC_CONF\", \"expandable_segments:True\")\n\nimport torch\n\nHERE = os.path.dirname(os.path.abspath(__file__))\n\n\ndef load_cfg(path: str) -> dict:\n    with open(path) as fp:\n        cfg = json.load(fp)\n    sft = cfg.get(\"sft\")\n    if not sft or \"lora_r\" not in sft:\n        raise SystemExit(f\"{path}: sft.lora_r missing \u2014 extend config first\")\n    return sft\n\n\ndef load_jsonl(path: str, limit: int = 0) -> list[dict]:\n    rows = []\n    with open(path) as fp:\n        for line in fp:\n            line = line.strip()\n            if line:\n                rows.append(json.loads(line))\n            if limit and len(rows) >= limit:\n                break\n    return rows\n\n\ndef build_dataset(rows: list[dict], tok, seq_len: int):\n    \"\"\"Chat template \u2192 input_ids/labels (R10: stable order as-is).\"\"\"\n    out_ids, out_masks, out_labels = [], [], []\n    for r in rows:\n        text = tok.apply_chat_template(r[\"messages\"], tokenize=False,\n                                       add_generation_prompt=False)\n        enc = tok(text, truncation=True, max_length=seq_len,\n                  add_special_tokens=False)\n        ids = enc[\"input_ids\"]\n        out_ids.append(ids)\n        out_masks.append(enc[\"attention_mask\"])\n        out_labels.append(list(ids))          # full-sequence SFT\n    return out_ids, out_masks, out_labels\n\n\nclass ListDataset:\n    \"\"\"Minimal torch-compatible dataset (no HF datasets dependency).\"\"\"\n\n    def __init__(self, ids, masks, labels):\n        self.ids, self.masks, self.labels = ids, masks, labels\n\n    def __len__(self):\n        return len(self.ids)\n\n    def __getitem__(self, i):\n        return {\"input_ids\": self.ids[i], \"attention_mask\": self.masks[i],\n                \"labels\": self.labels[i]}\n\n\nclass SFTCollator:\n    \"\"\"v6 (v5 incident): image Trainer routed variable-length labels through\n    tokenizer.pad -> ValueError. Collate ourselves: dynamic pad to batch\n    max, pads masked with -100, pure torch tensors (no tokenizer).\"\"\"\n\n    def __init__(self, pad_id: int):\n        self.pad = int(pad_id)\n\n    def __call__(self, feats):\n        import torch\n        m = max(len(f[\"input_ids\"]) for f in feats)\n        ii, am, lb = [], [], []\n        for f in feats:\n            n = len(f[\"input_ids\"])\n            ii.append(list(f[\"input_ids\"]) + [self.pad] * (m - n))\n            am.append(list(f[\"attention_mask\"]) + [0] * (m - n))\n            lb.append(list(f[\"labels\"]) + [-100] * (m - n))\n        return {\n            \"input_ids\": torch.tensor(ii, dtype=torch.long),\n            \"attention_mask\": torch.tensor(am, dtype=torch.long),\n            \"labels\": torch.tensor(lb, dtype=torch.long),\n        }\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--config\",\n                    default=os.path.join(HERE, \"config\", \"harness_config.json\"))\n    ap.add_argument(\"--data\", default=os.environ.get(\n        \"SFT_DATA\", os.path.join(HERE, \"..\", \"data\", \"sft\")))\n    ap.add_argument(\"--base\", default=os.environ.get(\"SFT_BASE\", \"\"))\n    ap.add_argument(\"--out\", default=os.environ.get(\"SFT_OUT\",\n                                                    os.path.join(HERE, \"adapter_r2\")))\n    ap.add_argument(\"--eval-samples\", type=int, default=500)\n    args = ap.parse_args()\n\n    sft = load_cfg(args.config)\n    base = args.base\n    if not base or not os.path.isdir(base):\n        raise SystemExit(f\"SFT_BASE not a directory: {base!r} \"\n                         \"(stage the model or set env)\")\n\n    import torch\n    from transformers import (AutoModelForCausalLM, AutoTokenizer,\n                              Trainer, TrainerCallback, TrainingArguments)\n    from peft import LoraConfig, TaskType, get_peft_model\n\n    seed = int(sft[\"seed\"])\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    precision = sft.get(\"precision\", \"auto\")\n    if precision == \"auto\":\n        bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()\n    else:\n        bf16 = precision == \"bf16\"\n    fp16 = torch.cuda.is_available() and not bf16\n    dev = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    print(f\"base={base} dev={dev} bf16={bf16} fp16={fp16}\", flush=True)\n\n    tok = AutoTokenizer.from_pretrained(base, trust_remote_code=True)\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    model = AutoModelForCausalLM.from_pretrained(\n        base,\n        torch_dtype=(torch.bfloat16 if bf16 else torch.float16)\n        if dev == \"cuda\" else torch.float32,\n        trust_remote_code=True)\n    model.config.use_cache = False\n\n    lora = LoraConfig(\n        task_type=TaskType.CAUSAL_LM,\n        r=int(sft[\"lora_r\"]), lora_alpha=int(sft[\"lora_alpha\"]),\n        lora_dropout=float(sft[\"lora_dropout\"]),\n        target_modules=list(sft[\"target_modules\"]),\n        bias=\"none\",\n    )\n    model = get_peft_model(model, lora)\n    # v7/v8: ckpt + LoRA need input grads flowing (PEFT does not enable\n    # this itself). v8 (v7 OOM with only 24.9MB fragmentation => real\n    # capacity, activations evidently uncheckpointed): FORCE ckpt at the\n    # model level regardless of what TrainingArguments claims to apply.\n    try:\n        model.gradient_checkpointing_enable(\n            gradient_checkpointing_kwargs={\"use_cache\": False})\n        model.config.use_cache = False\n        print(\"gradient checkpointing FORCED (use_cache=False)\")\n    except Exception as e:\n        print(\"ckpt force failed:\", e)\n    if hasattr(model, \"enable_input_require_grads\"):\n        model.enable_input_require_grads()\n    model.print_trainable_parameters()\n\n    train_path = os.path.join(args.data, \"train.jsonl\")\n    held_path = os.path.join(args.data, \"heldout.jsonl\")\n    tr_ids, tr_m, tr_l = build_dataset(load_jsonl(train_path), tok,\n                                       int(sft[\"seq_len\"]))\n    he_ids, he_m, he_l = build_dataset(load_jsonl(held_path, args.eval_samples),\n                                       tok, int(sft[\"seq_len\"]))\n    print(f\"train={len(tr_ids)} heldout_eval={len(he_ids)}\", flush=True)\n\n    # deterministic order (R10): jsonl is already written in sorted-tid order\n    train_ds = ListDataset(tr_ids, tr_m, tr_l)\n    eval_ds = ListDataset(he_ids, he_m, he_l)\n\n    out_dir = args.out\n    os.makedirs(out_dir, exist_ok=True)\n    t0 = time.time()\n\n    def _mem(tag):\n        try:\n            import torch as _t\n            a = _t.cuda.memory_allocated() / 2**30\n            r = _t.cuda.memory_reserved() / 2**30\n            print(f\"[mem] {tag}: allocated={a:.2f}GiB reserved={r:.2f}GiB\",\n                  flush=True)\n        except Exception as e:\n            print(f\"[mem] {tag}: n/a ({e})\")\n\n    _mem(\"after model+datasets\")\n    # v5 (v4 incident): image transformers dropped `warmup_ratio` from\n    # TrainingArguments. Filter kwargs by the image's real signature,\n    # preserving the ratified 3% warmup as explicit warmup_steps; any\n    # other removed kwarg is dropped LOUDLY (printed), never crashed.\n    import inspect\n    import math as _math\n    t_kwargs = dict(\n        output_dir=os.path.join(out_dir, \"ckpt\"),\n        num_train_epochs=float(sft[\"epochs\"]),\n        per_device_train_batch_size=int(sft[\"per_device_bs\"]),\n        gradient_accumulation_steps=int(sft[\"grad_accum\"]),\n        learning_rate=float(sft[\"lr\"]),\n        warmup_ratio=float(sft[\"warmup_ratio\"]),\n        lr_scheduler_type=\"cosine\",\n        logging_steps=25,\n        eval_strategy=\"epoch\",\n        # it1 incident (Oct19 run): save-at-end-only + 12h cell timeout =\n        # zero artifacts after ANY interruption. Persist every save_steps\n        # (save_total_limit keeps the dir small) so partial runs survive.\n        save_strategy=sft.get(\"save_strategy\", \"steps\"),\n        save_steps=int(sft.get(\"save_steps\", 250)),\n        save_total_limit=int(sft.get(\"save_total\", 2)),\n        bf16=bf16, fp16=fp16,\n        gradient_checkpointing=True,\n        # Oct19 incident: 43200s cell timeout with ZERO step output \u2014 the\n        # data is pre-tokenized in RAM (ListDataset), so workers buy\n        # nothing and forking AFTER CUDA init is a deadlock class of its\n        # own. v11: no workers.\n        dataloader_num_workers=int(sft.get(\"dataloader_workers\", 0)),\n        report_to=[],\n        seed=seed, data_seed=seed,\n        remove_unused_columns=False,\n    )\n    # optional smoke cap (diagnostic run): 50 steps then stop+save\n    _ms = int(sft.get(\"max_steps\", -1))\n    if _ms > 0:\n        t_kwargs[\"max_steps\"] = _ms\n    _params = inspect.signature(TrainingArguments.__init__).parameters\n    if \"warmup_ratio\" not in _params:\n        t_kwargs.pop(\"warmup_ratio\")\n        if \"warmup_steps\" in _params:\n            _total = _math.ceil(\n                len(tr_ids) / (int(sft[\"per_device_bs\"])\n                               * int(sft[\"grad_accum\"]))) * int(sft[\"epochs\"])\n            t_kwargs[\"warmup_steps\"] = max(\n                1, round(_total * float(sft[\"warmup_ratio\"])))\n            print(\"warmup_ratio absent -> warmup_steps:\",\n                  t_kwargs[\"warmup_steps\"], \"of\", _total)\n    _dropped = [k for k in list(t_kwargs) if k not in _params]\n    for k in _dropped:\n        t_kwargs.pop(k)\n    if _dropped:\n        print(\"TrainingArguments dropped (absent in image):\", _dropped)\n    args_t = TrainingArguments(**t_kwargs)\n    _pad = tok.pad_token_id\n    if _pad is None:\n        _pad = tok.eos_token_id\n    import gc as _gc\n    import torch as _torch\n    _torch.cuda.empty_cache()\n    _gc.collect()\n    # v11 (smoke incident): transformers calls EVERY callback event via\n    # getattr \u2014 duck-typing without the base raised AttributeError at\n    # on_train_begin. Subclass TrainerCallback (no-op defaults; immune to\n    # event-name drift across transformers versions).\n    class HeartbeatCallback(TrainerCallback):\n        \"\"\"Stdout step/loss/timing \u2014 the INFO logger never reaches the\n        Kaggle stream (Oct19: 12h of silence indistinguishable from death).\n        \"\"\"\n\n        def __init__(self, every: int = 25):\n            self.every = max(1, int(every))\n            self.t0 = time.time()\n            self.t_last = self.t0\n\n        def on_step_end(self, args_, state, control, **kw):\n            now = time.time()\n            dt = now - self.t_last\n            self.t_last = now\n            if state.global_step % self.every == 0 or state.global_step == 1:\n                loss = None\n                try:\n                    lh = state.log_history[-1]\n                    loss = lh.get(\"loss\", lh.get(\"eval_loss\"))\n                except Exception:\n                    pass\n                print(f\"[hb] step {state.global_step}/{state.max_steps} \"\n                      f\"loss={loss} step_dt={dt:.1f}s\", flush=True)\n\n        def on_save(self, args_, state, control, **kw):\n            print(f\"[hb] save @ step {state.global_step} -> \"\n                  f\"{args_.output_dir}\", flush=True)\n\n        def on_train_end(self, args_, state, control, **kw):\n            print(f\"[hb] train END step={state.global_step} \"\n                  f\"total={time.time() - self.t0:.0f}s\", flush=True)\n\n    trainer = Trainer(model=model, args=args_t,\n                      train_dataset=train_ds, eval_dataset=eval_ds,\n                      data_collator=SFTCollator(_pad),\n                      processing_class=tok)\n    trainer.add_callback(HeartbeatCallback(\n        5 if _ms > 0 else int(sft.get(\"heartbeat\", 25))))\n    _mem(\"before train\")\n    print(\"visible GPUs:\", torch.cuda.device_count(), flush=True)\n    used_seq = int(sft[\"seq_len\"])\n    # v12 (smoke incident Oct19): the v10 ladder HELD the OOM exception\n    # objects (e0 = e1 across iterations) \u2014 each traceback's frames pin the\n    # failed attempt's CUDA tensors (~5GiB), so every retry started with\n    # LESS headroom (7.5 \u2192 12.9 \u2192 14.1GiB before train) and 512/384 died\n    # although they fit clean. Keep only str(e); drop the exception\n    # (and its traceback \u2192 frames \u2192 tensors) BEFORE the teardown.\n    oom_msg = None\n    try:\n        res = trainer.train()\n    except torch.cuda.OutOfMemoryError as e:\n        oom_msg = str(e)\n        res = None\n        del e\n    if res is None:\n        for seq in (512, 384, 256):\n            if seq >= used_seq:\n                continue\n            print(f\"OOM at seq={used_seq} -> teardown + rebuild seq={seq}\",\n                  flush=True)\n            try:\n                del trainer\n            except Exception:\n                pass\n            _gc.collect()\n            _torch.cuda.empty_cache()\n            _torch.cuda.reset_peak_memory_stats()\n            tr_ids, tr_m, tr_l = build_dataset(load_jsonl(train_path),\n                                               tok, seq)\n            he_ids, he_m, he_l = build_dataset(\n                load_jsonl(held_path, args.eval_samples), tok, seq)\n            train_ds = ListDataset(tr_ids, tr_m, tr_l)\n            eval_ds = ListDataset(he_ids, he_m, he_l)\n            trainer = Trainer(model=model, args=args_t,\n                              train_dataset=train_ds, eval_dataset=eval_ds,\n                              data_collator=SFTCollator(_pad),\n                              processing_class=tok)\n            _mem(f\"before train (seq={seq})\")\n            try:\n                res = trainer.train()\n                used_seq = seq\n                break\n            except torch.cuda.OutOfMemoryError as e:\n                oom_msg = str(e)\n                res = None\n                del e\n        if res is None:\n            raise RuntimeError(\n                f\"OOM at every seq ladder attempt (512/384/256): {oom_msg}\")\n    _mem(\"after train\")\n    if used_seq != int(sft[\"seq_len\"]):\n        print(f\"[self-heal] trained at reduced seq={used_seq}\", flush=True)\n    # v12: eval is calibration, not a gate \u2014 an eval OOM must not kill an\n    # otherwise-saved run (the smoke lost its kernel AFTER train+save to\n    # exactly this). Null it loudly and continue.\n    held_loss = None\n    try:\n        held_loss = trainer.evaluate()[\"eval_loss\"]\n    except torch.cuda.OutOfMemoryError as e:\n        print(f\"[self-heal] eval OOM post-train \u2014 heldout_loss=null \"\n              f\"({e})\", flush=True)\n        del e\n        _gc.collect()\n        _torch.cuda.empty_cache()\n\n    trainer.model.save_pretrained(out_dir)\n    tok.save_pretrained(out_dir)\n    cfg_hash = hashlib.sha256(\n        open(args.config, \"rb\").read()).hexdigest()[:16]\n    meta = {\n        \"base\": base, \"config_hash\": cfg_hash,\n        \"lora_r\": sft[\"lora_r\"], \"lora_alpha\": sft[\"lora_alpha\"],\n        \"seq_len\": used_seq, \"epochs\": sft[\"epochs\"],\n        \"train_examples\": len(tr_ids), \"eval_samples\": len(he_ids),\n        \"train_loss\": float(res.training_loss),\n        # v12: null when the post-train eval OOMed (calibration skipped)\n        \"heldout_loss\": (float(held_loss)\n                         if held_loss is not None else None),\n        \"train_seconds\": round(time.time() - t0, 1),\n        \"precision\": \"bf16\" if bf16 else (\"fp16\" if fp16 else \"fp32\"),\n        \"gate\": sft[\"gate\"],\n    }\n    with open(os.path.join(out_dir, \"TRAIN_META.json\"), \"w\") as fp:\n        json.dump(meta, fp, indent=2)\n    print(json.dumps(meta, indent=2), flush=True)\n    print(f\"adapter saved: {out_dir}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n", "llm_serialize.py": "#!/usr/bin/env python3\n# design: projects/arc-prize/arc_prize_design.vsm\n\"\"\"llm_serialize \u2014 grid <-> text for LLM prompting (M1 harness).\n\nFormat (community grid-serialization style): every grid is rows of\ndigit characters (colors 0-9), one row per line. The prompt shows all\ntrain pairs as Input/Output blocks and asks for the test output grid\nONLY. Deterministic (R10): pure string transforms, no randomness.\n\"\"\"\nfrom __future__ import annotations\n\nfrom typing import List, Optional, Tuple\n\nGrid = List[List[int]]\nTrains = List[Tuple[Grid, Grid]]\n\n\ndef grid_to_text(g: Grid) -> str:\n    return \"\\n\".join(\"\".join(str(int(v)) for v in row) for row in g)\n\n\ndef grid_from_text(text: str) -> Optional[Grid]:\n    \"\"\"Parse the FIRST valid grid block from LLM text (robust).\"\"\"\n    rows: List[List[int]] = []\n    for raw in text.splitlines():\n        line = raw.strip().strip(\"`\").strip()\n        if not line:\n            if rows:\n                break\n            continue\n        # accept only pure digit rows (optionally space/comma separated)\n        cleaned = line.replace(\" \", \"\").replace(\",\", \"\")\n        if cleaned.isdigit() and len(cleaned) <= 64:\n            rows.append([int(ch) for ch in cleaned])\n            continue\n        if rows:\n            break\n    if not rows:\n        return None\n    width = len(rows[0])\n    if any(len(r) != width for r in rows):\n        return None\n    return rows\n\n\ndef build_prompt(trains: Trains, test: Grid) -> str:\n    parts = [\"Solve this abstract reasoning task. The test OUTPUT grid \"\n             \"follows the same rules as the train examples.\",\n             \"Output ONLY the test output grid: one row per line, digits \"\n             \"0-9, no explanation, no markdown.\"]\n    for i, (inp, out) in enumerate(trains, 1):\n        parts.append(f\"Train example {i} input:\\n{grid_to_text(inp)}\")\n        parts.append(f\"Train example {i} output:\\n{grid_to_text(out)}\")\n    parts.append(f\"Test input:\\n{grid_to_text(test)}\")\n    parts.append(\"Test output:\")\n    return \"\\n\\n\".join(parts)\n\n\ndef sanity_shape(pred: Grid, trains: Trains, max_factor: int = 4) -> bool:\n    \"\"\"Reject absurd shapes: at most max_factor x the largest train out.\"\"\"\n    if not pred:\n        return False\n    pr, pc = len(pred), len(pred[0])\n    if pr < 1 or pc < 1 or pr > 64 or pc > 64:\n        return False\n    max_r = max(len(o) for _i, o in trains)\n    max_c = max(len(o[0]) for _i, o in trains)\n    return pr <= max_r * max_factor and pc <= max_c * max_factor\n\n\nSYSTEM = (\"You are an expert at the ARC abstract reasoning benchmark. \"\n          \"You output ONLY grids as rows of digit characters.\")\n\n# \u2500\u2500 RSI-M2: tool-proposal mode (propose solve() code, not a grid) \u2500\u2500\u2500\u2500\u2500\u2500\n\nTOOL_SYSTEM = (\n    \"You are an expert ARC puzzle programmer. You write a single \"\n    \"Python function solve(trains, test) where trains is a list of \"\n    \"(input_grid, output_grid) pairs (lists of lists of ints 0-9) and \"\n    \"test is the test input grid. Return the predicted test output grid \"\n    \"or None. Use only: abs, all, any, bool, dict, enumerate, filter, \"\n    \"float, int, isinstance, len, list, map, max, min, range, round, \"\n    \"set, sorted, str, sum, tuple, type, zip. Output ONLY the Python \"\n    \"code inside one ```python block, no explanation.\")\n\n\ndef build_tool_prompt(trains: Trains, test: Grid) -> str:\n    \"\"\"Prompt asking for a PROGRAM that solves the task (RSI tool mode).\"\"\"\n    parts = [TOOL_SYSTEM, \"\"]\n    for i, (inp, out) in enumerate(trains, 1):\n        parts.append(f\"Train example {i} input:\\n{grid_to_text(inp)}\")\n        parts.append(f\"Train example {i} output:\\n{grid_to_text(out)}\")\n    parts.append(f\"Test input:\\n{grid_to_text(test)}\")\n    parts.append(\"Write solve(trains, test) as one python code block:\")\n    return \"\\n\\n\".join(parts)\n\n\ndef parse_tool_code(text: str) -> Optional[str]:\n    \"\"\"Extract the FIRST python code block from LLM text.\"\"\"\n    t = text or \"\"\n    if \"```python\" in t:\n        t = t.split(\"```python\", 1)[1]\n    elif \"```\" in t:\n        t = t.split(\"```\", 1)[1]\n    else:\n        return None\n    t = t.split(\"```\", 1)[0]\n    code = t.strip()\n    if not code or \"def solve\" not in code:\n        return None\n    if len(code) > 8000:  # absurdly long \u2014 refuse (sandbox safety)\n        return None\n    return code\n"}
CONFIG = "{\n \"_comment\": \"ViArc-2 harness config \u2014 single source of truth (visible on LXC 114 + kernel env). Plan: knowledge/viarc_master_plan.vsm\",\n \"system\": {\n  \"name\": \"ViArc-2\",\n  \"covenant\": \"knowledge/viarc_covenant.vsm (ratified 2026-10-16)\",\n  \"version\": \"m2-runtime\"\n },\n \"kernel_env\": {\n  \"LLM_K_SAMPLES\": \"4\",\n  \"LLM_LOO_MAX\": \"2\",\n  \"LLM_UNANIMOUS\": \"0.75\",\n  \"SMOKE_N\": \"0\",\n  \"_note\": \"v10 calibration: K=4+LOO=2 concentrated on the 240 scored test tasks (~82s x 240 = 5.5h < 9.5h guard); runtime decide_budget may shed to 0 under pressure (A0.1); ceiling K<=8 LOO<=2 (ratified)\"\n },\n \"covenant_ceiling\": {\n  \"certify_threshold\": 0.5,\n  \"K_max\": 8,\n  \"loo_max_task\": 2,\n  \"guard_h\": 9.5,\n  \"hard_deadline_h\": 12.0\n },\n \"models\": {\n  \"r0_current\": \"shuvocs23/qwen2-5-3b-instruct\",\n  \"r1_next_grid\": \"rmw3kufe/qwen3-4b-instruct\",\n  \"r1_next_tool\": \"rmw3kufe/qwen2.5-coder-3b-instruct\",\n  \"r2_sft_adapter\": null\n },\n \"sft\": {\n  \"method\": \"LoRA r=16 alpha=32 attn q/k/v/o fp16(T4) seq_len=512 lr=1e-4 calib-150\",\n  \"lora_r\": 16,\n  \"lora_alpha\": 32,\n  \"lora_dropout\": 0.05,\n  \"target_modules\": [\n   \"q_proj\",\n   \"k_proj\",\n   \"v_proj\",\n   \"o_proj\"\n  ],\n  \"seq_len\": 512,\n  \"_seq_note\": \"v12 (Oct19 smoke): 768 OOMed on T4 at loss upcast (1.16GiB logits.float); ladder validated 512 fits clean with ~5GiB headroom once the traceback-retention leak was fixed. Ratified intent stays 512 until an A100 slot exists.\",\n  \"lr\": 0.0001,\n  \"epochs\": 2,\n  \"per_device_bs\": 1,\n  \"grad_accum\": 32,\n  \"warmup_ratio\": 0.03,\n  \"precision\": \"fp16\",\n  \"_precision_note\": \"v11 (Oct19 incident): auto chose bf16=True on T4 (is_bf16_supported() lies on Turing); ratified intent = fp16 on T4/Turing (docstring sft_train). Forced.\",\n  \"dataloader_workers\": 0,\n  \"_workers_note\": \"v11: pre-tokenized RAM data \u2014 workers buy nothing; fork-after-CUDA is the Oct19 12h-silence deadlock class.\",\n  \"save_strategy\": \"steps\",\n  \"save_steps\": 50,\n  \"save_total\": 2,\n  \"_save_note\": \"v11: save-at-end-only lost every interrupted run; partial ckpts survive kills. v12 calib: 50 so the 150-step cap persists 3 ckpts (total=2 keeps last).\",\n  \"heartbeat\": 25,\n  \"max_steps\": 150,\n  \"_max_steps_note\": \"v12 CALIB (S5 2026-10-19, re-scope tras smoke): 150 steps ~5h train \u2014 adapter candidato + curva de loss + velocidad real. Corpus completo = post-entry (10-27..11-01), s\u00f3lo si el gate del calib pasa.\",\n  \"seed\": 42,\n  \"split\": \"by_task_hash_80_20\",\n  \"gate\": \"held_out_p1 >= zero_shot_p1 + 0.02\",\n  \"est_iter_hours\": \"CALIBRATED v12 (Oct19 smoke): ~65s/opt-step @ seq=256 on T4 \u2192 ~130s @ 512 \u2192 150 steps \u2248 5.5h train + ~40min setup. Full 1-epoch @512 \u2248 25h (multi-night, post-entry only).\"\n },\n \"calendar\": {\n  \"validation_kernel\": \"2026-10-16..18\",\n  \"sft_it1\": \"2026-10-17..20\",\n  \"sft_eval\": \"2026-10-21..22\",\n  \"v9_integrate\": \"2026-10-23..25\",\n  \"entry_deadline\": \"2026-10-26\",\n  \"final_deadline\": \"2026-11-02\"\n }\n}"
for name, src in MODULES.items():
    with open(name, "w") as fp:
        fp.write(src)
with open("harness_config.json", "w") as fp:
    fp.write(CONFIG)
print("modules:", sorted(MODULES))

# ── 3. locate staged base model ──────────────────────────────────
# v3: private kernels nest datasets/<owner>/<slug>/<files> (3 levels)
# — v2 logs proved it: /kaggle/input -> datasets -> rmw3kufe -> ...
# Walk with a depth cap; prefer the qwen3-4b base if several hit.
def find_base():
    hits = []
    for dp, dns, fns in os.walk("/kaggle/input"):
        if dp.count("/") - len("/kaggle/input".rstrip("/").split("/")) > 4:
            dns[:] = []
            continue
        fs = set(fns)
        if "model.safetensors.index.json" in fs or (
                "config.json" in fs and
                any(f.endswith(".safetensors") for f in fs)):
            hits.append(dp)
    for h in sorted(hits):
        if "qwen3-4b" in h:
            return h
    return sorted(hits)[0] if hits else None

print("listdir /kaggle/input:", sorted(os.listdir("/kaggle/input")))
base = find_base()
if not base:
    for _n in sorted(os.listdir("/kaggle/input"))[:20]:
        _d = os.path.join("/kaggle/input", _n)
        if os.path.isdir(_d):
            print("  ", _n, "->", sorted(os.listdir(_d))[:12])
    raise AssertionError("staged base model not found under /kaggle/input")
print("base:", base)

# ── 4. extract training tasks (challenges + solutions) ───────────
COMP = [p for p in glob.glob(
    "/kaggle/input/competitions/*") if "arc-agi-2" in p]
COMP = COMP[0] if COMP else None
assert COMP, "competition data mount not found"
chal = json.load(open(os.path.join(COMP,
                                   "arc-agi_training_challenges.json")))
sols = json.load(open(os.path.join(COMP,
                                   "arc-agi_training_solutions.json")))
tdir = "data/training"
os.makedirs(tdir, exist_ok=True)
for tid, task in chal.items():
    sol = sols.get(tid)
    if sol and len(sol) == len(task.get("test", [])):
        for t, s in zip(task["test"], sol):
            t["output"] = s
    with open(os.path.join(tdir, f"{tid}.json"), "w") as fp:
        json.dump(task, fp)
print("tasks extracted:", len(chal))

# ── 5. deterministic dataset build ───────────────────────────────
import sft_prepare
stats = sft_prepare.build(tdir, "data/sft", n_variants=7, seed=42)
print("sft stats:", json.dumps(stats["examples"]))

# ── 6. LoRA it1 ──────────────────────────────────────────────────
os.environ["SFT_BASE"] = base
os.environ["SFT_DATA"] = os.path.abspath("data/sft")
os.environ["SFT_OUT"] = os.path.abspath("adapter_r2")
sys.argv = ["sft_train.py", "--config", "harness_config.json"]
import sft_train
sft_train.main()
print("DONE adapter:", os.listdir("adapter_r2"))
